In [1]:
import pandas as pd
import numpy as np
import re
from datetime import timedelta
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Pattern regex per estrarre informazioni geografiche da stringhe di localizzazione (campo "place")
# Il pattern gestisce diverse componenti:
# - distanza in km e direzione (es. "10 km SE")
# - nome della città/località
# - eventuale sigla della provincia tra parentesi tonde (es. "(RM)")
LOCATION_PATTERN = re.compile(
    r"(?:(\d+)\s*kms?\s*([A-Za-z]+)\s+)?"
    r"\s*([^\(]+)\s*"
    r"(?:\(([^\)]+)\))?"
)

def parse_location(data: pd.DataFrame) -> pd.DataFrame:
    """
    Estrae e normalizza le informazioni geografiche dal campo 'place' di un DataFrame.
    
    Il campo 'place' è in genere una stringa che rappresenta la localizzazione di un evento
    (ad esempio sismico), e può contenere distanza, direzione, città e provincia:
    Esempio: "10 km SE Tivoli (RM)"

    Args: 
        data : pd.DataFrame
            DataFrame contenente almeno la colonna 'place'.

    Returns:
        pd.DataFrame
            Una copia del DataFrame originale con nuove colonne aggiunte:
            - 'city'      : nome della città o località
            - 'province'  : sigla della provincia (es. 'RM')
            - 'direction' : direzione cardinale (es. 'SE') se presente
            - 'km'        : distanza numerica in chilometri, se presente
    """
    # Applica il pattern regex alla colonna 'place' per estrarre le varie componenti
    matches = data["place"].str.extract(LOCATION_PATTERN)

    # Aggiunge le colonne estratte al DataFrame originale
    return data.assign(
        city=matches[2].str.strip(),                        # Nome città
        province=matches[3].str.strip().str[:2],            # Provincia (prime 2 lettere)
        direction=matches[1].str.upper(),                   # Direzione in maiuscolo (es. 'SE')
        km=pd.to_numeric(matches[0], errors="coerce")       # Km convertito in float, NaN se non parsabile
    )


In [ ]:
d = pd.read_csv("../data/terremoti_ingv_1985_2025.csv")
data = parse_location(d.copy())

In [4]:
data

,time,magnitude,place,depth,latitude,longitude,city,province,direction,km
0,1985-01-02T22:57:43.090000,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410,Civitella Alfedena,AQ,S,6.0
1,1985-01-03T00:46:16.990000,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310,Picinisco,FR,NE,6.0
2,1985-01-05T08:32:41.900000,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390,Fiastra,MC,NW,3.0
3,1985-01-05T10:13:26.800000,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600,San Raffaele Cimena,TO,E,1.0
4,1985-01-06T03:52:23.380000,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370,Gubbio,PG,W,3.0
...,...,...,...,...,...,...,...,...,...,...
441817,2024-12-31T22:00:34.100000,1.2,5 km NE Barberino di Mugello (FI),10.2,44.0287,11.2822,Barberino di Mugello,FI,NE,5.0
441818,2024-12-31T22:51:12.920000,1.1,4 km NE Barberino di Mugello (FI),10.3,44.0262,11.2815,Barberino di Mugello,FI,NE,4.0
441819,2024-12-31T23:29:28.530000,1.0,5 km NE Barberino di Mugello (FI),9.1,44.0267,11.2852,Barberino di Mugello,FI,NE,5.0
441820,2024-12-31T23:39:56.600000,0.7,4 km NE Barberino di Mugello (FI),10.1,44.0273,11.2777,Barberino di Mugello,FI,NE,4.0


In [5]:
data.describe()

,magnitude,depth,latitude,longitude,km
count,441822.000000,441822.000000,441822.000000,441822.000000,402824.000000
mean,1.514787,12.456870,42.489063,13.160691,4.100461
std,0.654107,18.976843,1.755783,1.629593,2.161051
min,0.000000,-0.300000,37.113300,6.229800,1.000000
25%,1.000000,8.200000,42.382800,12.630800,2.000000
50%,1.400000,10.000000,42.839200,13.134300,4.000000
75%,1.900000,11.300000,43.353300,13.431000,5.000000
max,6.500000,644.400000,47.891000,20.744000,42.000000


In [6]:
data.isna().sum()

time             0
magnitude        0
place            0
depth            0
latitude         0
longitude        0
city             0
province      3705
direction    38998
km           38998
dtype: int64

In [ ]:
def categorize_depth(depth):
    """
    Classifica la profondità di un evento (ad esempio sismico) in tre categorie:
    - "shallow" (superficiale)      : profondità < 70 km
    - "intermediate" (intermedia)   : 70 km ≤ profondità < 300 km
    - "deep" (profonda)             : profondità ≥ 300 km

    Args:
        depth : float or int
            La profondità in chilometri dell'evento.

    Returns:
        str
            Una stringa che rappresenta la categoria della profondità:
            "shallow", "intermediate" oppure "deep".
    """
    # Verifica se la profondità è minore di 70 km → evento superficiale
    if depth < 70:
        return "shallow"
    # Se è tra 70 e 299 km → evento intermedio
    elif depth < 300:
        return "intermediate"
    # Se ≥ 300 km → evento profondo
    else:
        return "deep"


In [ ]:
# Dizionario per convertire la direzione cardinale in gradi (sistema circolare trigonometricamente corretto)
direction_mapping = {
    "N": 0, "S": 180, "E": 90, "W": 270,
    "NE": 45, "NW": 315, "SE": 135, "SW": 225
}

# Mappa la direzione testuale in un angolo espresso in gradi
data["direction_angle"] = data["direction"].map(direction_mapping)

# Calcola la componente seno dell'angolo (utile per encoding circolare)
data["direction_angle_sin"] = np.sin(np.deg2rad(data["direction_angle"]))

# Calcola la componente coseno dell'angolo (complementare al seno per encoding circolare)
data["direction_angle_cos"] = np.cos(np.deg2rad(data["direction_angle"]))

In [ ]:
# Converte la colonna 'time' in formato datetime, gestendo eventuali errori di parsing con 'coerce'
data.loc[:, "time"] = pd.to_datetime(
    data["time"],
    format='%Y-%m-%dT%H:%M:%S.%f',
    errors='coerce',  # inserisce NaT se la stringa non è conforme al formato
    exact=True
)

# Converte la colonna in formato datetime64 standard di pandas (ridondante ma garantisce coerenza di tipo)
data["time"] = data["time"].astype("datetime64[ns]")

# Rimuove le righe con valori nulli nelle colonne essenziali per l'analisi
data = data.dropna(subset=["province", "direction", "km", "direction_angle"])

In [10]:
data

,time,magnitude,place,depth,latitude,longitude,city,province,direction,km,direction_angle,direction_angle_sin,direction_angle_cos
0,1985-01-02 22:57:43.090,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410,Civitella Alfedena,AQ,S,6.0,180.0,1.224647e-16,-1.000000e+00
1,1985-01-03 00:46:16.990,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310,Picinisco,FR,NE,6.0,45.0,7.071068e-01,7.071068e-01
2,1985-01-05 08:32:41.900,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390,Fiastra,MC,NW,3.0,315.0,-7.071068e-01,7.071068e-01
3,1985-01-05 10:13:26.800,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600,San Raffaele Cimena,TO,E,1.0,90.0,1.000000e+00,6.123234e-17
4,1985-01-06 03:52:23.380,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370,Gubbio,PG,W,3.0,270.0,-1.000000e+00,-1.836970e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...
441817,2024-12-31 22:00:34.100,1.2,5 km NE Barberino di Mugello (FI),10.2,44.0287,11.2822,Barberino di Mugello,FI,NE,5.0,45.0,7.071068e-01,7.071068e-01
441818,2024-12-31 22:51:12.920,1.1,4 km NE Barberino di Mugello (FI),10.3,44.0262,11.2815,Barberino di Mugello,FI,NE,4.0,45.0,7.071068e-01,7.071068e-01
441819,2024-12-31 23:29:28.530,1.0,5 km NE Barberino di Mugello (FI),9.1,44.0267,11.2852,Barberino di Mugello,FI,NE,5.0,45.0,7.071068e-01,7.071068e-01
441820,2024-12-31 23:39:56.600,0.7,4 km NE Barberino di Mugello (FI),10.1,44.0273,11.2777,Barberino di Mugello,FI,NE,4.0,45.0,7.071068e-01,7.071068e-01


In [ ]:
# === Feature engineering temporali ===

data = data.assign(
    # Estrae l'anno dalla data
    year=data["time"].dt.year,
    # Estrae il mese dalla data
    month=data["time"].dt.month,
    # Estrae il giorno del mese
    day=data["time"].dt.day,
    # Estrae il giorno della settimana come stringa (es. 'Monday')
    weekday=data["time"].dt.day_name(),
    # Estrae l'ora dell'evento
    hour=data["time"].dt.hour,
    # Crea un flag binario: 1 se è notte (dalle 22:00 alle 6:00), altrimenti 0
    is_night=((data["time"].dt.hour < 6) | (data["time"].dt.hour >= 22)).astype(int),
    # Crea una nuova variabile categorica per la stagione
    season=data["time"].dt.month.map({
        12: "winter", 1: "winter", 2: "winter",
        3: "spring", 4: "spring", 5: "spring",
        6: "summer", 7: "summer", 8: "summer",
        9: "autumn", 10: "autumn", 11: "autumn"
    })
)

In [ ]:
# === Altre feature numeriche e categoriali ===

# Aggiunge il quadrato della distanza in km come feature per modelli non lineari
data["km_squared"] = data["km"] ** 2

# Categorizza la profondità in 'shallow', 'intermediate', 'deep'
data["depth_category"] = data["depth"].apply(categorize_depth)

# === Encoding ciclico per variabili temporali ===

# Codifica ciclicamente il mese per rappresentare la periodicità dell’anno
data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

# Codifica ciclicamente l’ora del giorno per rappresentare la periodicità giornaliera
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

# === Feature derivate dalle coordinate geografiche ===

# Converte latitudine e longitudine da gradi a radianti
data["latitude_rad"] = np.deg2rad(data["latitude"])
data["longitude_rad"] = np.deg2rad(data["longitude"])

# Calcola seno e coseno della latitudine
data["lat_cos"] = np.cos(data["latitude_rad"])
data["lat_sin"] = np.sin(data["latitude_rad"])

# Calcola seno e coseno della longitudine
data["lon_cos"] = np.cos(data["longitude_rad"])
data["lon_sin"] = np.sin(data["longitude_rad"])

In [ ]:
# === Suddivisione temporale del dataset ===

# Ordina i dati cronologicamente
data = data.sort_values("time")

# Calcola l'indice di split (85% per sviluppo, 15% per test)
split_idx = int(len(data) * 0.85)

# Suddivide il dataset in dati di sviluppo (train+val) e test
dev_data, test_data = data.iloc[:split_idx], data.iloc[split_idx:]

In [ ]:
def add_dynamic_features(df, end_date):
    """
    Calcola feature dinamiche basate sull'attività sismica di ciascuna provincia
    in finestre temporali recenti (ultimi 30 e 7 giorni).

    Args:
        df : pd.DataFrame
            DataFrame contenente i dati sismici con almeno le colonne:
            - 'time' (datetime)
            - 'province' (str)
            - 'magnitude' (float)
            - 'km' (float) distanza
        end_date : datetime
            Data di fine per il calcolo delle finestre temporali (esclusa)

    Returns:
        pd.DataFrame
            DataFrame indicizzato per provincia con le seguenti colonne:
            - mag_mean_30d: magnitudo media negli ultimi 30 giorni
            - mag_trend_30d: coefficiente angolare della regressione lineare
                            della magnitudo negli ultimi 30 giorni (trend)
            - activity_ratio: rapporto tra numero eventi ultimi 7 giorni / ultimi 30 giorni
            - last_event_km: distanza (km) dell'ultimo evento registrato per provincia
    """
    # Calcola le date di inizio per le finestre mobili da 30 e 7 giorni
    start_date_30d = end_date - timedelta(days=30)
    start_date_7d = end_date - timedelta(days=7)
    
    # Filtra i dati per eventi compresi nelle ultime 30 e 7 giorni
    df_30d = df[df["time"].between(start_date_30d, end_date)]
    df_7d = df[df["time"].between(start_date_7d, end_date)]
    
    province_stats = {}
    # Itera su ogni provincia unica nei dati
    for province in df["province"].unique():
        # Dati completi per la provincia
        province_data = df[df["province"] == province]
        # Dati ultimi 30 giorni per la provincia
        province_30d = df_30d[df_30d["province"] == province]
        # Dati ultimi 7 giorni per la provincia
        province_7d = df_7d[df_7d["province"] == province]
        
        # Calcola le statistiche dinamiche richieste
        stats = {
            # Magnitudo media ultimi 30 giorni (NaN se nessun evento)
            "mag_mean_30d": province_30d["magnitude"].mean(),
            
            # Trend magnitudo ultimi 30 giorni tramite regressione lineare
            # Se meno di 2 eventi, trend = 0 (nessuna pendenza calcolabile)
            "mag_trend_30d": LinearRegression().fit(
                np.arange(len(province_30d)).reshape(-1, 1),
                province_30d["magnitude"]
            ).coef_[0] if len(province_30d) > 1 else 0,
            
            # Rapporto tra numero eventi ultimi 7 giorni e ultimi 30 giorni
            # Evita divisione per zero se ultimi 30 giorni = 0 eventi
            "activity_ratio": len(province_7d) / len(province_30d) if len(province_30d) > 0 else 0,
            
            # Distanza (km) dell'ultimo evento nella provincia (NaN se dati vuoti)
            "last_event_km": province_data["km"].iloc[-1] if not province_data.empty else np.nan
        }
        
        # Salva le statistiche nel dizionario con chiave la provincia
        province_stats[province] = stats
    
    # Restituisce un DataFrame con righe per provincia e colonne delle feature calcolate
    return pd.DataFrame.from_dict(province_stats, orient='index')


In [ ]:
# Codifica ciclica del mese per rappresentare la periodicità annuale
data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

# Codifica ciclica dell'ora per rappresentare la periodicità giornaliera
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)


# Conversione delle coordinate geografiche da gradi a radianti
data["latitude_rad"] = np.deg2rad(data["latitude"])
data["longitude_rad"] = np.deg2rad(data["longitude"])

# Calcolo di seno e coseno della latitudine per rappresentazioni circolari
data["lat_cos"] = np.cos(data["latitude_rad"])
data["lat_sin"] = np.sin(data["latitude_rad"])

# Calcolo di seno e coseno della longitudine per rappresentazioni circolari
data["lon_cos"] = np.cos(data["longitude_rad"])
data["lon_sin"] = np.sin(data["longitude_rad"])


In [16]:
data.isna().sum()

time                   0
magnitude              0
place                  0
depth                  0
latitude               0
longitude              0
city                   0
province               0
direction              0
km                     0
direction_angle        0
direction_angle_sin    0
direction_angle_cos    0
year                   0
month                  0
day                    0
weekday                0
hour                   0
is_night               0
season                 0
km_squared             0
depth_category         0
month_sin              0
month_cos              0
hour_sin               0
hour_cos               0
latitude_rad           0
longitude_rad          0
lat_cos                0
lat_sin                0
lon_cos                0
lon_sin                0
dtype: int64

In [ ]:
# Generazione delle feature dinamiche per il set di sviluppo (dev)
# end_date_dev prende la data massima presente nel dev_data
end_date_dev = dev_data["time"].max()
# Calcolo delle feature dinamiche basate sui dati fino a end_date_dev
dynamic_features_dev = add_dynamic_features(dev_data, end_date_dev)
# Fusione delle feature dinamiche con il dataset di sviluppo, unendo per provincia
dev_data = dev_data.merge(dynamic_features_dev, left_on="province", right_index=True)

In [19]:
dev_data.isna().sum()

time                      0
magnitude                 0
place                     0
depth                     0
latitude                  0
longitude                 0
city                      0
province                  0
direction                 0
km                        0
direction_angle           0
direction_angle_sin       0
direction_angle_cos       0
year                      0
month                     0
day                       0
weekday                   0
hour                      0
is_night                  0
season                    0
km_squared                0
depth_category            0
month_sin                 0
month_cos                 0
hour_sin                  0
hour_cos                  0
latitude_rad              0
longitude_rad             0
lat_cos                   0
lat_sin                   0
lon_cos                   0
lon_sin                   0
mag_mean_30d           8119
mag_trend_30d             0
activity_ratio            0
last_event_km       

In [20]:
dev_data = dev_data.dropna(subset=["mag_mean_30d"])

In [ ]:
# Generazione delle feature dinamiche per il set di test, usando solo dati storici del dev set
# end_date_test è impostato come la data massima del dev_data per evitare "data leakage"
end_date_test = dev_data["time"].max()  
# Calcolo delle feature dinamiche per il test set usando solo dati storici (dev_data)
dynamic_features_test = add_dynamic_features(dev_data, end_date_test)
# Fusione delle feature dinamiche con il test set, usando left join per mantenere tutte le righe del test set
test_data = test_data.merge(dynamic_features_test, left_on="province", right_index=True, how='left')

In [22]:
test_data.isna().sum()

time                      0
magnitude                 0
place                     0
depth                     0
latitude                  0
longitude                 0
city                      0
province                  0
direction                 0
km                        0
direction_angle           0
direction_angle_sin       0
direction_angle_cos       0
year                      0
month                     0
day                       0
weekday                   0
hour                      0
is_night                  0
season                    0
km_squared                0
depth_category            0
month_sin                 0
month_cos                 0
hour_sin                  0
hour_cos                  0
latitude_rad              0
longitude_rad             0
lat_cos                   0
lat_sin                   0
lon_cos                   0
lon_sin                   0
mag_mean_30d           1955
mag_trend_30d          1955
activity_ratio         1955
last_event_km       

In [23]:
test_data = test_data.dropna(subset=["mag_mean_30d", "mag_trend_30d", "activity_ratio", "last_event_km"])

In [ ]:
dev_data.to_csv("../data/processed/dev_data.csv", index=False)
test_data.to_csv("../data/processed/test_data.csv", index=False)